### Data Cleaning and Lemmatization 

This is the code used to perform data cleaning and lemmatization for the skill and knowledge extraction pipeline. With some operations being specific to a given data, make sure to investigate your data before applying these methods. 

In [7]:
#Import all the required libraries 
import pandas as pd
import spacy
import regex as re
import nltk
from urllib.parse import urlparse, urlsplit
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.tokenize import PunktSentenceTokenizer
from nltk.corpus import wordnet as wn
nltk.download('stopwords') #for stopwords
nltk.download('punkt')# for punctation
nltk.download('averaged_perceptron_tagger') #for tagging the words based on their speech type
nltk.download('wordnet') #for lemmanizer

[nltk_data] Downloading package stopwords to C:\Users\Aleksander
[nltk_data]     Bielinski\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Aleksander
[nltk_data]     Bielinski\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Aleksander
[nltk_data]     Bielinski\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to C:\Users\Aleksander
[nltk_data]     Bielinski\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
#Import your data to a pandas dataframe format
'''
The data used for this could look something like this (see below). 
The essential component is "JobText" consisting of job description text, the rest could be omitted at this stage: 
'''
data = {
    'Employer': ['DPD Group', None, 'Brewdog Plc', 'Heineken UK Limited', 'Belhaven Brewery Company Limited'],
    'City': ['Glasgow', 'Turriff', 'Peterhead', 'Edinburgh', None],
    'County/UA': ['Glasgow City', 'Aberdeenshire', 'Aberdeenshire', 'City of Edinburgh', 'North Lanarkshire'],
    'Title': ['Shift Manager', 'Slaugther/Butcher', 'Assistant Manager - Brewdog', 'Commerce Solutions Architect', 'General Manager Designate'],
    'JobDate': ['2019-01-03', '2019-01-03', '2019-01-03', '2019-01-04', '2019-01-04'],
    'SIC CODE': ['01.30', '10.11', '11.05', '11.05', '11.05'],
    'SOC CODE': ['1121', '5431', '3520', '2135', '1223'],
    'JobText': ['Shift Manager Location: Glasgow Eurocentral Po...', '* Search * Previous Housing Assistant * Cook -...', 'Assistant Manager - BrewDog PeterheadBrewDog a...', 'Location : Edinburgh (EH) - Midlothian, Scotla...', 'General Manager Designate General Manager Desi...'],
    'JobUrl': ['careers.dpd.co.uk', 'www.jobsinscotland.com', 'www.brewdog.com', 'www.tiptopjob.com', 'www.fish4.co.uk'],
    'DescriptionLenght': [486, 197, 374, 685, 456]
}

df = pd.DataFrame(data)

In [5]:
#This function cleans most of the patterns that were not relevant to the skill and knowledge extraction task. 
#These patterns were identified heuristically, through manual investigation of the 50 job descriptions. 

def cleaning(text):
    text_01 = re.sub(r'\t', '', text)
    text_02 = re.sub(r'\n', '',  text_01)
    text_1 = re.sub(r'\s+', ' ', text_02)
    text_2 = re.sub(r'&amp;', '&', text_1)
    text_2_01 = re.sub(r'&apos;', ' ', text_2)
    text_3 = re.sub(r'(?=(TRY\s?{)).*?(?<=};)', '', text_2_01, flags=re.IGNORECASE)
    text_3_01_01 = re.sub(r'(\.CSS.*)', '', text_3, flags=re.IGNORECASE)
    text_3_01 = re.sub(r'(COOKIE\s+NOTICE).*', '', text_3_01_01, flags=re.IGNORECASE)
    text_3_02 = re.sub(r'\b(N / A|NA |NULL |NAN |N/A)\W', '', text_3_01, flags=re.IGNORECASE)
    text_3_03 = re.sub(r'.*(useAutoScrolling).*', '', text_3_02, flags=re.IGNORECASE)
    text_4 = re.sub(r'\b\d{1,2}\s+\w+\s+\d{2,4}\W', ' ', text_3_03)
    text_4_5 = re.sub(r'\s+', ' ', text_4)
    text_5 = re.sub(r'\w*[/]\w*\S*\d.', '', text_4_5)
    text_6 = re.sub(r'http?\S*|www?\S*', '', text_5)
    text_7 = re.sub(r'\S*\.com|\S*\.uk', '', text_6)
    text_8 = re.sub(r'\b(EMAIL|E-MAIL)?\W+\S*[@]\S*\W', '', text_7, flags=re.IGNORECASE)
    text_8_5 = re.sub(r'\b(JOB)?\W?SALARY\W+(\£+\W?|\?+\W?)?(COMPETITIVE|NEGOTIABLE)\W|\b(JOB)?\W?SALARY\W+(FROM)?\W+(\£+\W?|\?+\W?)?\W?(\d{1,6})((\.|\,)\d{2,6})?\W?(\sTO\s|W+)?\W+(\d{1,4})((\.|\,)\d{2,4})?\W?', ' ', text_8, flags=re.IGNORECASE)
    text_8_6 = re.sub(r'\s+', ' ', text_8_5)
    text_9 = re.sub(r'\b(THE)?\W+(WORKING|WORK)?\W+(HOURS|HOUR|SCHEDULE)\W+(ARE|IS)?\W+(FROM|BETWEEN)?\W+(\d{1,4})(\.\d{2})?\W?(PM|AM)?(\W+|\s(AND|TO)\s)(\d{1,4})(\.\d{2})?\W?(PM|AM)?\W|\b(DEPENDING|DEPENDS|DEPEND)\W+ON\W+(THE)?\W+(EXPERIENCE)\W|\b(EXPERIENCE)\W+(DEPENDANT|DEPEND)\W|\b(P/H|WEEAKLY|P/W|P/Y|YEARLY|ANNUALLY)\W|\b(PER)\W+(YEAR|WEEK|DAY|ANNUM)\W|\b(PH)(\s+|-+|:+|,+|\.+|$)', ' ', text_8_6, flags=re.IGNORECASE)
    text_9_5 = re.sub(r'\s+', ' ', text_9)
    text_10 = re.sub(r'\b(LOCATION|SECTOR)\W+\w*\W|\b(CONTACT)\W+\w*\s?\w*\W|\b(PUBLISHED|PUBLISH)\W?(ABOUT)?\W+\d{1,2}\W+(HOURS|HOUR|YESTERDAY|DAYS|DAY)\W?(AGO)?\W|\b(JOB)?\W?(TITLE|NAME|REFERENCE|REF\.?)\W?(CODE|NUMBER|NO\.?)?\W?|\b(DURATION)\W?(\w*|\d{1,2})\W?(DAYS|WEEKS|MONTHS|DAY|WEEK|MONTH)?\W|\b(STARTDATE|START)\W+(\w*)\W', ' ', text_9_5, flags=re.IGNORECASE)
    text_10_5 = re.sub(r'\b(FIRST|LAST)\W+(NAME)\W', ' ', text_10, flags = re.IGNORECASE)
    text_10_6 = re.sub(r'\b(PLEASE)\W+(SELECT)\W+', ' ', text_10_5, flags = re.IGNORECASE)
    text_10_7 = re.sub(r'\s+', ' ', text_10_6)
    text_11 = re.sub(r'\bTELEPHONE\W+\d{2,}.|\bCALL\W+\d{2,}.|\bPHONE\W+\d{2,}.', '', text_10_7, flags=re.IGNORECASE)
    text_12 = re.sub(r'\S*[|]\S*', '', text_11)
    text_13 = re.sub(r'\bPAY\W+RATE\W|\b(WORKING)\W+(SCHEDULE)\W|\b(OVERTIME)\W+(LOCATION)\W|\b(ADDITIONAL)?\W+(BENEFITS|BENEFIT)\W?(INCLUDE)?\W', ' ', text_12, flags=re.IGNORECASE)
    text_13_05 = re.sub(r'\s+', ' ', text_13)
    text_14 = re.sub(r'\bFULL\W+TIME\W?(CONTRACT|JOB)?\W|\bPART\W+TIME\W?(CONTRACT|JOB)?\W|\bFIXED\W+TERM\W?(CONTRACT|JOB)?\W', '', text_13_05, flags=re.IGNORECASE)
    text_15 = re.sub(r'\b(£|$|\?|\/)?\W+\d{2,}(\s+-?\+?|-+|\/+|\.+|$)', '', text_14)
    text_15_5 = re.sub(r'(--)\W*', '', text_15)
    text_16 = re.sub(r'\b(DATE|JUST)\W+(POSTED|POST)\W|\bSTART\W+DATE\W|\b(JOB|CONTRACT)\W?TYPE?\W?(TEMPORARY|PERMANENT)?\W|\bCONTRACT\W+LENGTH\W|\bAPPLY\W+(NOW|HERE|ON)\W|\bTO\W+APPLY\W|\bCLICK?\W+APPLY\W?(NOW)?\W|\bPLEASE?\W?COMPLETE\W?(THE|YOUR)?\W+APPLICATION\W?(FORM)?\W|\bEMPLOYMENT\W+AGENCY\W|\b(CARRERS|CARRER)\W+(AGENCY|WEBSITE)\W|\b(ONLY)?\W?(\d{1,})?\W+(HOURS|HOUR)\W?(A|PER)?\W+(DAY|WEEK)\W?(ONLY)?\W|\bORIGINAL\W+(JOB)\W?|\bREPORT\W+(JOB)\W|(\d{1,})?\W+(DAYS|DAY)\W+AGO\W|\bDAY\W+HOLIDAY\W|\b(DOWNLOADABLE|DOWNLOAD)\W+(HERE|THERE|AT)\W|\b(CLOSING|CLOSE)\W+DATE\W?(FOR|TO)?\W?(APPLICATIONS|APPLICATION|APPLY)?\W|\b(JOB)\W+(TYPE|LOCATION)\W|\b(JOB|VACANCY)\W+(POSTED|POST)\W|(POSTED|POST)\W+ON\W|\bEMPLOYMENT\W+HOURS\W+(FULL|PART)?\W|\bBACK?\W+TO\W+RESULTS\W|\b(E\.MAIL|E-MAIL|EMAIL)\W?(JOB)?\W+TO\W?A?\W+FRIEND\W|\bFIND?\W+SIMILAR\W+(JOBS|JOB)\W', ' ', text_15_5, flags=re.IGNORECASE)
    text_16_05 = re.sub(r'\d{2,}\S*.', ' ', text_16)
    text_17 = re.sub(r'\s+', ' ', text_16_05)
    return text_17

### Lemmatization

This section focuses on assessing the lemmatization performance of five different algorithms, namely Wordnet Lemmatized, Wordnet Lemmatized with (POS) tag, Spacy (small, large and transformer-based). The lemmatization testing was applied before any other data cleaning operations. 

In [8]:
#Select the random subset of your data (I suggest 50)
#Example df_sample = df.sample(n=50, random_seed=x)
#This function skips unnecessary stages of the pipeline

#Lemmatization for Spacy models

nlp = spacy.load('en_core_web_lg') # for large
#nlp = spacy.load('en_core_web_sm') # for small
#nlp = spacy.load('en_core_web_sm') # for transformer based one
 
def spacylemm_lang(text):
  sentence = nlp(text)
  tokens = []
  for token in sentence:
    tokens.append(token)

    lemmatized_sentence = " ".join([token.lemma_ for token in sentence])

  return lemmatized_sentence

#Lemmatization for WordNet

def lemmatize(text):
    lem = WordNetLemmatizer()

    words_tokenized = nltk.word_tokenize(text)
    
    lemmatized_string = ' '.join([lem.lemmatize(words) for words in words_tokenized])
    
    return(lemmatized_string)

#with POS tags

def pos_tagger(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:         
        return None

def lemmatize_pos(text):
   lem = WordNetLemmatizer()
   pos_tagged = nltk.pos_tag(nltk.word_tokenize(text))
   wordnet_tagged = list(map(lambda x: (x[0], pos_tagger(x[1])), pos_tagged))
   lemmatized_sentence = []
   for word, tag in wordnet_tagged:
    if tag is None:
        # if there is no available tag, append the token as is
        lemmatized_sentence.append(word)
    else:       
        # else use the tag to lemmatize the token
        lemmatized_sentence.append(lem.lemmatize(word, tag))
   lemmatized_string = " ".join(lemmatized_sentence)
   return lemmatized_string
   

In [9]:
'''
The lemmatizer function, make sure to select the right algorithm number. alg = 1 (spacy), alg = 2 (wordnet_base), alg = 3 (wordnet_pos_tag). 
Also ensure you update the function with correct column names. 
'''
def lemmatize_test(data, alg = 1):
    
    #lemmatization of the texts (spacy- make sure the desired model is selected)
    if alg == 1:

        new_dataframe = pd.DataFrame()
        new_dataframe['Lemmatized'] = data['JobText'].apply(lambda x: spacylemm_lang(x))
        ew_dataframe = pd.concat([data, new_dataframe], axis=1, join='outer')
        new_dataframe = new_dataframe.drop(labels=['JobText'], axis=1)

    if alg == 2:

        new_dataframe = pd.DataFrame()
        new_dataframe['Lemmatized'] = data['JobText'].apply(lambda x: lemmatize(x))
        new_dataframe = pd.concat([data, new_dataframe], axis=1, join='outer')
        new_dataframe = new_dataframe.drop(labels=['JobText'], axis=1)

    if alg == 3:

        new_dataframe = pd.DataFrame()
        new_dataframe['Lemmatized'] = data['JobText'].apply(lambda x: lemmatize_pos(x))
        new_dataframe = pd.concat([data, new_dataframe], axis=1, join='outer')
        new_dataframe = new_dataframe.drop(labels=['JobText'], axis=1)

    return new_dataframe
    

### Analysis of the Lemmatization

After you lemmatize your sample with each of the proposed models, export the dataframes to excel/csv or preferred format. That way you will be able to investigate the performance more clearly. Note that in this case the Spacy's large english model delivered the best overall results concerning efficiency and efficacy.

### Final Cleaning

After selecting the right lemmatizer for your use case, the next step it to perform cleaning on a large scale. While initially the full text were lemmatized, considering the potentially large size of the sample, it is important to perform all the operations that might reduce the length of text and the size of the sample. For that reason the duplicates and common phrases (see cleaning function) were removed. To ensure proper functionality of the lemmatizer, stopwords were removed post lemmatization. 

### Duplicate Removal
In a job posting database it is possible that some of the jobs will be posted overtime using the same information. The same applies to different regions.
However, given that the project want to model the labour market demand, removing all of those would not be optimal. The key is to remove the exact same job postings, ensuring that they were posted in different regions or date. 
While this still is likely to retain some jobs that simply couldn't be filled, such process would allow for keep the same job postings posted in different regions and/or time, thus capturing the true demand. 
In short, only the the same job postings scraped from different websites were removed. Some experiments were done to compare the sample size before and after variations of duplicates removed, including:


In [11]:
#Duplicate removal

#df_1 = df.drop_duplicates(['JobText','City'],keep= 'last')
#df_2 = df.drop_duplicates(['JobText','JobDate'],keep= 'last')
#df_3 = df.drop_duplicates(['JobText'],keep= 'last')

#The final choice was to remove duplicates that were posted on the same date in the same region and the ones that regions, title, city and job text matched.

#df = df.drop_duplicates(['JobText', 'JobDate', 'County/UA'],keep= 'last')
#df = df.drop_duplicates(['JobText', 'city', 'title'],keep= 'last')

In [12]:
#Defining the stopset, remember to download one before using nltk.download('stopwords') 
stopset_r = set(stopwords.words('english'))


#adding some stop words to the stop set based on the initial findings

#adding "on"

stopset_r.update(['on','approximately', 'approximately', 'jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec', 'january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november', 'december', ':', 'love', 'please', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'overtime', '()', '( )', 'bonus',' bonuses', '£' , '?', 'be', 'aberdeen', 'armagh', 'bangor', 'bangor', 'bath', 'belfast', 'birmingham', 'bradford', 'brighton hove', 'bristol', 'cambridge', 'canterbury', 'cardiff', 'carlisle', 'chelmsford', 'chester', 'chichester', 'colchester', 'coventry', 'derby', 'doncaster', 'dundee', 'dunfermline', 'durham', 'edinburgh', 'ely', 'exeter', 'glasgow', 'gloucester', 'hereford', 'inverness', 'kingston upon hull', 'lancaster', 'leeds', 'leicester', 'lichfield', 'lincoln', 'lisburn', 'liverpool', 'london', 'londonderry', 'manchester', 'milton keynes', 'newcastle upon tyne', 'newport', 'newry', 'norwich', 'nottingham', 'oxford', 'perth', 'peterborough', 'plymouth', 'portsmouth', 'preston', 'ripon', 'salford', 'salisbury', 'sheffield', 'southampton', 'southend - on - sea', 'st albans', 'st asaph ', 'llanelwy', 'st davids', 'stirling', 'stoke-on-trent', 'sunderland', 'swansea', 'truro', 'wakefield', 'wells', 'westminster', 'winchester', 'wolverhampton', 'worcester', 'wrexham', 'york', 'bedfordshire', 'berkshire', 'bristol', 'buckinghamshire', 'cambridgeshire', 'cheshire', 'cornwall', 'cumbria', 'derbyshire', 'devon', 'dorset', 'durham', 'east riding of yorkshire', 'east sussex', 'essex', 'gloucestershire', 'greater london', 'greater manchester', 'hampshire', 'herefordshire', 'hertfordshire', 'isle of wight', 'kent', 'lancashire', 'leicestershire', 'lincolnshire', 'merseyside', 'middlesex', 'norfolk', 'north yorkshire', 'northamptonshire', 'northumberland', 'nottinghamshire', 'oxfordshire', 'rutland', 'shropshire', 'somerset', 'south yorkshire', 'staffordshire', 'suffolk', 'surrey', 'tyne and wear', 'warwickshire', 'west midlands', 'west sussex', 'west yorkshire', 'wiltshire', 'worcestershire', 'county antrim', 'county armagh', 'county down', 'county fermanagh', 'county londonderry', 'county tyrone', 'aberdeen', 'aberdeenshire', 'angus', 'argyll and bute', 'clackmannanshire', 'dumfries and galloway', 'dundee', 'east ayrshire', 'east dunbartonshire', 'east lothian', 'east renfrewshire', 'edinburgh', 'falkirk', 'fife', 'glasgow', 'highland', 'inverclyde', 'midlothian', 'moray', 'north ayrshire', 'north lanarkshire', 'orkney', 'perth and kinross', 'renfrewshire', 'scottish borders', 'shetland isles', 'south ayrshire', 'south lanarkshire', 'stirlingshire', 'west dunbartonshire', 'west lothian', 'western isles', 'anglesey / sir fon', 'anglesey/sir fon', 'blaenau gwent', 'bridgend', 'caerphilly', 'cardiff', 'carmarthenshire', 'ceredigion', 'conwy', 'denbighshire', 'flintshire', 'glamorgan', 'gwynedd', 'merthyr tydfil', 'monmouthshire', 'neath port talbot', 'newport', 'newport city', 'pembrokeshire', 'powys', 'rhondda cynon taff', 'swansea', 'torfaen', 'wrexha'])

#removing (no, not, nor)

#stopset_r.remove(['no', 'not', 'nor'])

stopset_r.remove('during')
stopset_r.remove('about')
stopset_r.remove('other')
stopset_r.remove('off')
stopset_r.remove('from')
stopset_r.remove('down')
stopset_r.remove('under')
stopset_r.remove('over')
stopset_r.remove('it')

In [13]:
#Stopwords removal. 
'''
The standard English stopset was annotated with phrases that were identified as redundant during the lemmatizer selection. 
Additionally some potentially relevant keywords were removed from the stopset. 
'''
def stopwords_rem(text):
    
    stopwords = stopset_r
    
    full_text = text.lower()
    words_st_0 = re.sub(r'[^\w\s|^/|^\-{1}|\.{1}|:{1}|+|#]', '',  full_text)
    words_st_1 = re.sub(r'(?<!C )(?<!C)(?<!C\+)(?<!C\+ )(?<!C \+)(?<!C \+ )(?<!c )(?<!c)(?<!c\+)(?<!c\+ )(?<!c \+)(?<!c \+ )\+', '', words_st_0)
    words_st_2 = re.sub(r'(?<!C )(?<!C)(?<!C\+)(?<!C\+ )(?<!C \+)(?<!C \+ )(?<!c )(?<!c)(?<!c\+)(?<!c\+ )(?<!c \+)(?<!c \+ )\#', '', words_st_1)
    words_st = re.sub(r'\bit\b', '', words_st_2).split()
    tokens = []
    for token in words_st:
        if token not in stopwords:
            tokens.append(token)
    return " ".join(tokens)

In [14]:
#With the duplicates removed it was time to prepare the sample for the topic-modeling 
def clean_text(data): 
    
    #Regular Expressions
    new_dataframe = pd.DataFrame()
    new_dataframe['Cleaned'] = data['JobText'].apply(lambda x: cleaning(x))
    new_dataframe = pd.concat([data, new_dataframe], axis=1, join='outer')
    new_dataframe = new_dataframe.drop(labels=['JobText'], axis=1)
    
    #lemmatization of the texts
    new_dataframe_1 = pd.DataFrame()
    new_dataframe_1['lemmatized_cleared'] = new_dataframe['Cleaned'].apply(lambda x: spacylemm_lang(x))
    new_dataframe_1 = pd.concat([new_dataframe, new_dataframe_1], axis=1, join='outer')
    new_dataframe_1 = new_dataframe_1.drop(labels=['Cleaned'], axis=1)
    
    #stopwords
    new_dataframe_2 = pd.DataFrame()
    new_dataframe_2['Tokens'] = new_dataframe_1['lemmatized_cleared'].apply(lambda x: stopwords_rem(x))
    new_dataframe_2 = pd.concat([new_dataframe_1, new_dataframe_2], axis=1, join='outer')
    new_dataframe_2 = new_dataframe_2.drop(labels=['lemmatized_cleared'], axis=1)
    
    new_dataframe_2 = new_dataframe_2.reset_index(drop=True)
    
    return new_dataframe_2

In [17]:
#The last part is to apply the clean_text function to all datasets (remember to update the function with your column names)
#Afterwards your data will be ready for the topic modelling. 
df_clean = clean_text(df)